In [ ]:
import dlt
from pyspark.sql.functions import (
    col, when, sum, count, avg, max, min, stddev, countDistinct, first, last,
    coalesce, lit, current_date, current_timestamp, date_sub, date_add, months_between,
    year, month, quarter, dayofmonth, dayofweek, weekofyear, date_format,
    lag, lead, row_number, rank, dense_rank, ntile, percent_rank, cume_dist,
    concat, concat_ws, round, abs, sqrt, log, exp, sin, cos,
    regexp_extract, split, substring, length, trim, upper, lower,
    collect_list, collect_set, array_contains, array_size, explode,
    struct, map_values, map_keys, size, sort_array, reverse, datediff, greatest
)
from pyspark.sql.types import DecimalType, IntegerType, DateType, StringType, DoubleType
from pyspark.sql.window import Window
import pyspark.sql.functions as F


In [ ]:
# ===============================================================================
#                           ENTERPRISE GOLD LAYER CONFIGURATION
# ===============================================================================

env = spark.conf.get("pipeline.env") 
catalog = "raiffka_catalog"
silver_schema = f"{env}_silver" if env else "silver"
gold_schema = f"{env}_gold" if env else "gold"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {gold_schema}")

# Business constants
CURRENT_DATE = current_date()
CURRENT_TIMESTAMP = current_timestamp()
LOOKBACK_PERIODS = {
    "7d": 7,
    "30d": 30, 
    "90d": 90,
    "180d": 180,
    "365d": 365
}


In [ ]:
# ===============================================================================
#                              ENTERPRISE FACT TABLES
# ===============================================================================

#------------------------------- CUSTOMER LIFETIME VALUE FACT -------------------------
@dlt.table(
    name="fact_customer_lifetime_value",
    comment="Enterprise fact table tracking customer lifetime value with sophisticated financial metrics",
    table_properties={
        "quality": "gold",
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true"
    },
    partition_cols=["calculation_year_month"]
)
def fact_customer_lifetime_value():
    # Source tables
    customers = dlt.readStream(f"{catalog}.{silver_schema}.dim_customers")
    accounts = dlt.readStream(f"{catalog}.{silver_schema}.dim_accounts")
    transactions = dlt.readStream(f"{catalog}.{silver_schema}.fact_transactions_enriched")
    balances = dlt.readStream(f"{catalog}.{silver_schema}.account_balances_silver")
    risk_metrics = dlt.readStream(f"{catalog}.{silver_schema}.customer_risk_metrics_silver")
    
    # Calculate comprehensive customer metrics over multiple time periods
    customer_financial_metrics = (
        transactions
        .filter(col("transaction_date") >= date_sub(CURRENT_DATE, LOOKBACK_PERIODS["365d"]))
        .groupBy("customer_id")
        .agg(
            # Revenue metrics
            sum(when(col("amount") > 0, col("amount")).otherwise(0)).alias("total_inflow_365d"),
            sum(when(col("amount") < 0, abs(col("amount"))).otherwise(0)).alias("total_outflow_365d"),
            count("*").alias("total_transactions_365d"),
            avg("amount").alias("avg_transaction_amount_365d"),
            
            # Behavioral metrics
            countDistinct("merchant_category").alias("category_diversity_365d"),
            countDistinct(date_format("transaction_date", "yyyy-MM-dd")).alias("active_days_365d"),
            
            # Seasonal patterns
            sum(when(quarter(col("transaction_date")) == 1, col("amount")).otherwise(0)).alias("q1_amount"),
            sum(when(quarter(col("transaction_date")) == 2, col("amount")).otherwise(0)).alias("q2_amount"),
            sum(when(quarter(col("transaction_date")) == 3, col("amount")).otherwise(0)).alias("q3_amount"),
            sum(when(quarter(col("transaction_date")) == 4, col("amount")).otherwise(0)).alias("q4_amount"),
            
            # Recent activity (last 30 days)
            sum(when(col("transaction_date") >= date_sub(CURRENT_DATE, 30), col("amount")).otherwise(0)).alias("recent_net_flow_30d"),
            count(when(col("transaction_date") >= date_sub(CURRENT_DATE, 30), 1)).alias("recent_transactions_30d")
        )
    )
    
    # Calculate average balance metrics
    customer_balance_metrics = (
        balances
        .filter(col("datum") >= date_sub(CURRENT_DATE, LOOKBACK_PERIODS["365d"]))
        .join(accounts.filter(col("__END_AT").isNull()), "account_id", "inner")
        .groupBy("customer_id")
        .agg(
            avg("balance").alias("avg_balance_365d"),
            max("balance").alias("max_balance_365d"),
            min("balance").alias("min_balance_365d"),
            stddev("balance").alias("balance_volatility_365d"),
            count("*").alias("balance_observations_365d")
        )
    )
    
    return (
        customers.alias("c")
        .filter(col("c.__END_AT").isNull())  # Current customers only
        .join(customer_financial_metrics.alias("fm"), col("c.customer_id") == col("fm.customer_id"), "left")
        .join(customer_balance_metrics.alias("bm"), col("c.customer_id") == col("bm.customer_id"), "left")
        .join(risk_metrics.alias("rm"), col("c.customer_id") == col("rm.customer_id"), "left")
        .select(
            col("c.customer_id"),
            col("c.jmeno"),
            col("c.prijmeni"),
            col("c.client_age"),
            col("c.prijem").alias("annual_income"),
            col("c.days_from_open_first_acc").alias("customer_tenure_days"),
            
            # Financial performance metrics
            coalesce(col("fm.total_inflow_365d"), lit(0)).alias("total_inflow_365d"),
            coalesce(col("fm.total_outflow_365d"), lit(0)).alias("total_outflow_365d"),
            (coalesce(col("fm.total_inflow_365d"), lit(0)) - coalesce(col("fm.total_outflow_365d"), lit(0))).alias("net_cash_flow_365d"),
            coalesce(col("fm.total_transactions_365d"), lit(0)).alias("total_transactions_365d"),
            coalesce(col("fm.avg_transaction_amount_365d"), lit(0)).alias("avg_transaction_amount_365d"),
            
            # Balance metrics
            coalesce(col("bm.avg_balance_365d"), lit(0)).alias("avg_balance_365d"),
            coalesce(col("bm.max_balance_365d"), lit(0)).alias("max_balance_365d"),
            coalesce(col("bm.min_balance_365d"), lit(0)).alias("min_balance_365d"),
            coalesce(col("bm.balance_volatility_365d"), lit(0)).alias("balance_volatility_365d"),
            
            # Behavioral insights
            coalesce(col("fm.category_diversity_365d"), lit(0)).alias("spending_category_diversity"),
            coalesce(col("fm.active_days_365d"), lit(0)).alias("active_days_365d"),
            (coalesce(col("fm.active_days_365d"), lit(0)) / 365.0).alias("activity_ratio"),
            
            # Seasonal spending patterns
            coalesce(col("fm.q1_amount"), lit(0)).alias("q1_spending"),
            coalesce(col("fm.q2_amount"), lit(0)).alias("q2_spending"),
            coalesce(col("fm.q3_amount"), lit(0)).alias("q3_spending"),
            coalesce(col("fm.q4_amount"), lit(0)).alias("q4_spending"),
            
            # Recent activity indicators
            coalesce(col("fm.recent_net_flow_30d"), lit(0)).alias("recent_net_flow_30d"),
            coalesce(col("fm.recent_transactions_30d"), lit(0)).alias("recent_transactions_30d"),
            
            # Risk and profitability scores
            coalesce(col("rm.composite_risk_score"), lit(5)).alias("risk_score"),
            coalesce(col("rm.risk_category"), lit("Unknown")).alias("risk_category"),
            
            # Calculated CLV components
            when(coalesce(col("bm.avg_balance_365d"), lit(0)) > 100000, "Premium")
            .when(coalesce(col("bm.avg_balance_365d"), lit(0)) > 50000, "Gold")
            .when(coalesce(col("bm.avg_balance_365d"), lit(0)) > 20000, "Silver")
            .otherwise("Standard").alias("customer_tier"),
            
            # Revenue potential (simplified CLV calculation)
            (
                (coalesce(col("bm.avg_balance_365d"), lit(0)) * 0.02) +  # 2% interest margin
                (coalesce(col("fm.total_transactions_365d"), lit(0)) * 0.5) +  # Transaction fees
                (col("c.prijem") * 0.001)  # Income-based potential
            ).alias("estimated_annual_revenue"),
            
            # Churn risk indicators
            when(coalesce(col("fm.recent_transactions_30d"), lit(0)) == 0, "High Churn Risk")
            .when(coalesce(col("fm.recent_transactions_30d"), lit(0)) < 5, "Medium Churn Risk")
            .otherwise("Low Churn Risk").alias("churn_risk_level"),
            
            # Metadata
            date_format(CURRENT_DATE, "yyyy-MM").alias("calculation_year_month"),
            CURRENT_TIMESTAMP.alias("calculated_at")
        )
        .withColumn("lifetime_value_score",
                   round((col("estimated_annual_revenue") * (col("customer_tenure_days") / 365.0)) / 
                         (1 + col("risk_score")), 2))
    )


In [ ]:
#------------------------------- ADVANCED TRANSACTION ANALYTICS FACT -------------------------
@dlt.table(
    name="fact_transaction_analytics_advanced",
    comment="Advanced transaction analytics with fraud detection, pattern analysis, and predictive insights",
    table_properties={
        "quality": "gold",
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true"
    },
    partition_cols=["transaction_year", "transaction_month"]
)
def fact_transaction_analytics_advanced():
    transactions = dlt.readStream(f"{catalog}.{silver_schema}.fact_transactions_enriched")
    customers = dlt.readStream(f"{catalog}.{silver_schema}.dim_customers")
    geography = dlt.read(f"{catalog}.{silver_schema}.dim_geography_hierarchy")
    
    # Define window specifications for advanced analytics
    customer_window = Window.partitionBy("customer_id").orderBy("transaction_date")
    customer_7d_window = Window.partitionBy("customer_id").orderBy("transaction_date").rangeBetween(-7*86400, 0)
    customer_30d_window = Window.partitionBy("customer_id").orderBy("transaction_date").rangeBetween(-30*86400, 0)
    
    merchant_window = Window.partitionBy("merchant_id").orderBy("transaction_date")
    
    return (
        transactions.alias("t")
        .join(customers.filter(col("__END_AT").isNull()).alias("c"), 
              col("t.customer_id") == col("c.customer_id"), "inner")
        .join(geography.alias("g"), col("t.city_name") == col("g.city_name"), "left")
        .select(
            col("t.transaction_id"),
            col("t.customer_id"),
            col("t.account_id"),
            col("t.merchant_id"),
            col("t.transaction_type"),
            col("t.amount"),
            col("t.transaction_date"),
            col("t.merchant_category"),
            col("t.transaction_value_segment"),
            col("t.transaction_hour"),
            col("t.transaction_day_of_week"),
            col("t.transaction_month"),
            col("t.transaction_year"),
            
            # Customer context
            col("c.client_age"),
            col("c.prijem").alias("customer_income"),
            col("c.days_from_open_first_acc").alias("customer_tenure"),
            
            # Geography context
            coalesce(col("g.region_name"), lit("Unknown")).alias("transaction_region"),
            coalesce(col("g.city_name"), lit("Unknown")).alias("transaction_city"),
            
            # Advanced transaction patterns
            lag(col("t.amount"), 1).over(customer_window).alias("prev_transaction_amount"),
            lag(col("t.transaction_date"), 1).over(customer_window).alias("prev_transaction_date"),
            lead(col("t.amount"), 1).over(customer_window).alias("next_transaction_amount"),
            lead(col("t.transaction_date"), 1).over(customer_window).alias("next_transaction_date"),
            
            # Rolling aggregations for behavior analysis
            sum(col("t.amount")).over(customer_7d_window).alias("amount_sum_7d"),
            count("*").over(customer_7d_window).alias("transaction_count_7d"),
            avg(col("t.amount")).over(customer_7d_window).alias("amount_avg_7d"),
            stddev(col("t.amount")).over(customer_7d_window).alias("amount_stddev_7d"),
            
            sum(col("t.amount")).over(customer_30d_window).alias("amount_sum_30d"),
            count("*").over(customer_30d_window).alias("transaction_count_30d"),
            avg(col("t.amount")).over(customer_30d_window).alias("amount_avg_30d"),
            max(col("t.amount")).over(customer_30d_window).alias("amount_max_30d"),
            min(col("t.amount")).over(customer_30d_window).alias("amount_min_30d"),
            
            # Transaction sequencing
            row_number().over(customer_window).alias("customer_transaction_sequence"),
            rank().over(Window.partitionBy("customer_id", "merchant_category").orderBy("transaction_date")).alias("category_transaction_rank"),
            
            # Merchant analytics
            count("*").over(merchant_window).alias("merchant_total_transactions"),
            avg(col("t.amount")).over(merchant_window).alias("merchant_avg_transaction"),
            
            CURRENT_TIMESTAMP.alias("calculated_at")
        )
        .withColumn("days_since_prev_transaction", 
                   when(col("prev_transaction_date").isNotNull(), 
                        datediff(col("transaction_date"), col("prev_transaction_date")))
                   .otherwise(lit(None)))
        .withColumn("amount_vs_prev_pct_change",
                   when(col("prev_transaction_amount").isNotNull() & (col("prev_transaction_amount") != 0),
                        round((col("amount") - col("prev_transaction_amount")) / col("prev_transaction_amount") * 100, 2))
                   .otherwise(lit(None)))
        .withColumn("amount_vs_7d_avg_deviation",
                   when(col("amount_avg_7d").isNotNull() & (col("amount_stddev_7d") > 0),
                        round((col("amount") - col("amount_avg_7d")) / col("amount_stddev_7d"), 2))
                   .otherwise(lit(0)))
        .withColumn("is_weekend_transaction",
                   when(col("transaction_day_of_week").isin([1, 7]), lit(1)).otherwise(lit(0)))
        .withColumn("is_night_transaction", 
                   when(col("transaction_hour").between(22, 6), lit(1)).otherwise(lit(0)))
                 .withColumn("transaction_velocity_score",
                   when(col("days_since_prev_transaction") <= 1, 5)  # High velocity
                   .when(col("days_since_prev_transaction") <= 3, 4)
                   .when(col("days_since_prev_transaction") <= 7, 3)
                   .when(col("days_since_prev_transaction") <= 14, 2)
                   .otherwise(1))  # Low velocity
        .withColumn("anomaly_score",
                   (
                       when(abs(col("amount_vs_7d_avg_deviation")) > 3, 3).otherwise(0) +  # Statistical outlier
                       when(col("is_night_transaction") == 1, 1).otherwise(0) +  # Night transaction
                       when(col("amount") > col("amount_max_30d") * 1.5, 2).otherwise(0) +  # Unusually high amount
                       when(col("transaction_velocity_score") >= 4, 1).otherwise(0)  # High velocity
                   ))
                 .withColumn("fraud_risk_level",
                   when(col("anomaly_score") >= 5, "High Risk")
                   .when(col("anomaly_score") >= 3, "Medium Risk") 
                   .when(col("anomaly_score") >= 1, "Low Risk")
                   .otherwise("Normal"))
                 .withColumn("customer_behavior_segment",
                   when((col("transaction_count_30d") >= 50) & (col("amount_avg_30d") > 1000), "High Value High Frequency")
                   .when((col("transaction_count_30d") >= 50) & (col("amount_avg_30d") <= 1000), "Low Value High Frequency")
                   .when((col("transaction_count_30d") < 10) & (col("amount_avg_30d") > 1000), "High Value Low Frequency")
                   .otherwise("Standard"))
    )


In [ ]:
# ===============================================================================
#                           ENTERPRISE AGGREGATION TABLES
# ===============================================================================

#------------------------------- DAILY BUSINESS INTELLIGENCE SUMMARY -------------------------
@dlt.table(
    name="agg_daily_business_intelligence",
    comment="Daily business intelligence aggregations for executive dashboards and KPI monitoring",
    table_properties={
        "quality": "gold",
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true"
    },
    partition_cols=["business_date_year", "business_date_month"]
)
def agg_daily_business_intelligence():
    transactions = dlt.readStream(f"{catalog}.{silver_schema}.fact_transactions_enriched")
    customers = dlt.readStream(f"{catalog}.{silver_schema}.dim_customers")
    balances = dlt.readStream(f"{catalog}.{silver_schema}.account_balances_silver")
    accounts = dlt.readStream(f"{catalog}.{silver_schema}.dim_accounts")
    geography = dlt.read(f"{catalog}.{silver_schema}.dim_geography_hierarchy")
    
    # Daily transaction aggregations
    daily_transaction_metrics = (
        transactions
        .filter(col("transaction_date") >= date_sub(CURRENT_DATE, 90))  # Last 90 days
        .groupBy(date_format("transaction_date", "yyyy-MM-dd").alias("business_date"))
        .agg(
            # Volume metrics
            count("*").alias("total_transactions"),
            countDistinct("customer_id").alias("active_customers"),
            countDistinct("merchant_id").alias("active_merchants"),
            
            # Value metrics
            sum("amount").alias("total_transaction_value"),
            avg("amount").alias("avg_transaction_value"),
            max("amount").alias("max_transaction_value"),
            min("amount").alias("min_transaction_value"),
            stddev("amount").alias("transaction_value_stddev"),
            
            # Transaction type breakdown
            sum(when(col("transaction_type") == "CARD", col("amount")).otherwise(0)).alias("card_transaction_value"),
            sum(when(col("transaction_type") == "FINANCIAL", col("amount")).otherwise(0)).alias("financial_transaction_value"),
            count(when(col("transaction_type") == "CARD", 1)).alias("card_transaction_count"),
            count(when(col("transaction_type") == "FINANCIAL", 1)).alias("financial_transaction_count"),
            
            # Risk metrics
            count(when(col("transaction_value_segment") == "High Value", 1)).alias("high_value_transactions"),
            sum(when(col("transaction_value_segment") == "High Value", col("amount")).otherwise(0)).alias("high_value_transaction_amount"),
            
            # Time-based patterns
            count(when(col("transaction_hour").between(9, 17), 1)).alias("business_hours_transactions"),
            count(when(col("transaction_hour").between(18, 23), 1)).alias("evening_transactions"),
            count(when(col("transaction_hour").between(0, 8), 1)).alias("night_transactions"),
            
            # Geographic distribution
            countDistinct("city_name").alias("unique_cities_transacting"),
            countDistinct("region_name").alias("unique_regions_transacting")
        )
    )
    
    # Daily balance aggregations (using latest balance per account per day)
    daily_balance_metrics = (
        balances
        .filter(col("datum") >= date_sub(CURRENT_DATE, 90))
        .join(accounts.filter(col("__END_AT").isNull()), "account_id", "inner")
        .join(customers.filter(col("__END_AT").isNull()), "customer_id", "inner")
        .groupBy(date_format("datum", "yyyy-MM-dd").alias("business_date"))
        .agg(
            # Balance metrics
            sum("balance").alias("total_customer_balances"),
            avg("balance").alias("avg_customer_balance"),
            max("balance").alias("max_customer_balance"),
            min("balance").alias("min_customer_balance"),
            count("*").alias("total_account_balance_records"),
            countDistinct("customer_id").alias("customers_with_balances"),
            
            # Balance distribution
            count(when(col("balance") > 100000, 1)).alias("high_balance_accounts"),
            count(when(col("balance").between(50000, 100000), 1)).alias("medium_balance_accounts"),
            count(when(col("balance").between(10000, 50000), 1)).alias("low_balance_accounts"),
            count(when(col("balance") < 10000, 1)).alias("minimal_balance_accounts"),
            count(when(col("balance") < 0, 1)).alias("negative_balance_accounts"),
            
            # Balance concentration
            sum(when(col("balance") > 100000, col("balance")).otherwise(0)).alias("high_balance_total_value"),
            sum(when(col("balance") < 0, abs(col("balance"))).otherwise(0)).alias("negative_balance_total_value")
        )
    )
    
    return (
        daily_transaction_metrics.alias("tm")
        .join(daily_balance_metrics.alias("bm"), col("tm.business_date") == col("bm.business_date"), "full_outer")
        .select(
            coalesce(col("tm.business_date"), col("bm.business_date")).alias("business_date"),
            
            # Transaction KPIs
            coalesce(col("tm.total_transactions"), lit(0)).alias("total_transactions"),
            coalesce(col("tm.active_customers"), lit(0)).alias("active_customers"),
            coalesce(col("tm.active_merchants"), lit(0)).alias("active_merchants"),
            coalesce(col("tm.total_transaction_value"), lit(0)).alias("total_transaction_value"),
            coalesce(col("tm.avg_transaction_value"), lit(0)).alias("avg_transaction_value"),
            
            # Balance KPIs
            coalesce(col("bm.total_customer_balances"), lit(0)).alias("total_customer_balances"),
            coalesce(col("bm.avg_customer_balance"), lit(0)).alias("avg_customer_balance"),
            coalesce(col("bm.customers_with_balances"), lit(0)).alias("customers_with_balances"),
            
            # Business ratios and insights
            round(coalesce(col("tm.total_transaction_value"), lit(0)) / 
                  greatest(coalesce(col("tm.total_transactions"), lit(1)), lit(1)), 2).alias("revenue_per_transaction"),
            round(coalesce(col("tm.total_transaction_value"), lit(0)) / 
                  greatest(coalesce(col("tm.active_customers"), lit(1)), lit(1)), 2).alias("revenue_per_active_customer"),
            
            # Risk indicators
            coalesce(col("tm.high_value_transactions"), lit(0)).alias("high_value_transactions"),
            coalesce(col("tm.high_value_transaction_amount"), lit(0)).alias("high_value_transaction_amount"),
            coalesce(col("bm.negative_balance_accounts"), lit(0)).alias("negative_balance_accounts"),
            coalesce(col("bm.negative_balance_total_value"), lit(0)).alias("negative_balance_total_value"),
            
            # Channel distribution
            coalesce(col("tm.card_transaction_count"), lit(0)).alias("card_transaction_count"),
            coalesce(col("tm.financial_transaction_count"), lit(0)).alias("financial_transaction_count"),
            coalesce(col("tm.card_transaction_value"), lit(0)).alias("card_transaction_value"),
            coalesce(col("tm.financial_transaction_value"), lit(0)).alias("financial_transaction_value"),
            
            # Time pattern analysis
            coalesce(col("tm.business_hours_transactions"), lit(0)).alias("business_hours_transactions"),
            coalesce(col("tm.evening_transactions"), lit(0)).alias("evening_transactions"),
            coalesce(col("tm.night_transactions"), lit(0)).alias("night_transactions"),
            
            # Geographic reach
            coalesce(col("tm.unique_cities_transacting"), lit(0)).alias("unique_cities_transacting"),
            coalesce(col("tm.unique_regions_transacting"), lit(0)).alias("unique_regions_transacting"),
            
            # Balance distribution insights
            coalesce(col("bm.high_balance_accounts"), lit(0)).alias("high_balance_accounts"),
            coalesce(col("bm.medium_balance_accounts"), lit(0)).alias("medium_balance_accounts"),
            coalesce(col("bm.low_balance_accounts"), lit(0)).alias("low_balance_accounts"),
            coalesce(col("bm.minimal_balance_accounts"), lit(0)).alias("minimal_balance_accounts"),
            
            # Calculated dimensions for partitioning
            year(coalesce(col("tm.business_date"), col("bm.business_date"))).alias("business_date_year"),
            month(coalesce(col("tm.business_date"), col("bm.business_date"))).alias("business_date_month"),
            
            CURRENT_TIMESTAMP.alias("calculated_at")
        )
        .withColumn("balance_concentration_ratio",
                   round(coalesce(col("high_balance_total_value"), lit(0)) / 
                         greatest(coalesce(col("total_customer_balances"), lit(1)), lit(1)), 4))
        .withColumn("transaction_efficiency_score",
                   round((coalesce(col("total_transactions"), lit(0)) * coalesce(col("avg_transaction_value"), lit(0))) / 
                         greatest(coalesce(col("active_customers"), lit(1)), lit(1)), 2))
                 .withColumn("risk_indicator_score",
                   when((col("negative_balance_accounts") > 100) | (col("high_value_transactions") > 1000), "High")
                   .when((col("negative_balance_accounts") > 50) | (col("high_value_transactions") > 500), "Medium") 
                   .otherwise("Low"))
    )


In [ ]:
#------------------------------- CUSTOMER SEGMENTATION ADVANCED -------------------------
@dlt.table(
    name="agg_customer_segmentation_advanced",
    comment="Advanced customer segmentation with RFM analysis, behavioral clustering, and predictive scoring",
    table_properties={
        "quality": "gold",
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true"
    }
)
def agg_customer_segmentation_advanced():
    customers = dlt.readStream(f"{catalog}.{silver_schema}.dim_customers")
    transactions = dlt.readStream(f"{catalog}.{silver_schema}.fact_transactions_enriched")
    balances = dlt.readStream(f"{catalog}.{silver_schema}.account_balances_silver")
    accounts = dlt.readStream(f"{catalog}.{silver_schema}.dim_accounts")
    risk_metrics = dlt.readStream(f"{catalog}.{silver_schema}.customer_risk_metrics_silver")
    
    # RFM Analysis (Recency, Frequency, Monetary)
    rfm_analysis = (
        transactions
        .filter(col("transaction_date") >= date_sub(CURRENT_DATE, 365))
        .groupBy("customer_id")
        .agg(
            # Recency: Days since last transaction
            datediff(CURRENT_DATE, max("transaction_date")).alias("recency_days"),
            
            # Frequency: Number of transactions
            count("*").alias("frequency_transactions"),
            countDistinct(date_format("transaction_date", "yyyy-MM-dd")).alias("frequency_active_days"),
            
            # Monetary: Total transaction value
            sum(abs("amount")).alias("monetary_total_value"),
            avg(abs("amount")).alias("monetary_avg_transaction"),
            max(abs("amount")).alias("monetary_max_transaction"),
            
            # Advanced behavioral metrics
            countDistinct("merchant_category").alias("category_diversity"),
            countDistinct("merchant_id").alias("merchant_diversity"),
            
            # Seasonal behavior
            sum(when(month("transaction_date").isin([12, 1, 2]), abs("amount")).otherwise(0)).alias("winter_spending"),
            sum(when(month("transaction_date").isin([3, 4, 5]), abs("amount")).otherwise(0)).alias("spring_spending"),
            sum(when(month("transaction_date").isin([6, 7, 8]), abs("amount")).otherwise(0)).alias("summer_spending"),
            sum(when(month("transaction_date").isin([9, 10, 11]), abs("amount")).otherwise(0)).alias("autumn_spending"),
            
            # Time pattern behavior
            count(when(col("transaction_hour").between(9, 17), 1)).alias("business_hours_txns"),
            count(when(col("transaction_hour").between(18, 23), 1)).alias("evening_txns"),
            count(when(col("transaction_hour").between(0, 8), 1)).alias("night_txns"),
            
            # Channel preference
            count(when(col("transaction_type") == "CARD", 1)).alias("card_transactions"),
            count(when(col("transaction_type") == "FINANCIAL", 1)).alias("financial_transactions")
        )
    )
    
    # Balance behavior analysis
    balance_behavior = (
        balances
        .filter(col("datum") >= date_sub(CURRENT_DATE, 365))
        .join(accounts.filter(col("__END_AT").isNull()), "account_id", "inner")
        .groupBy("customer_id")
        .agg(
            avg("balance").alias("avg_balance_365d"),
            stddev("balance").alias("balance_volatility"),
            max("balance").alias("max_balance"),
            min("balance").alias("min_balance"),
            count(when(col("balance") < 0, 1)).alias("negative_balance_days"),
            count("*").alias("total_balance_observations")
        )
    )
    
    # Calculate RFM scores and percentiles
    rfm_with_scores = (
        rfm_analysis
        .withColumn("recency_percentile", percent_rank().over(Window.orderBy(col("recency_days"))))
        .withColumn("frequency_percentile", percent_rank().over(Window.orderBy(desc("frequency_transactions"))))
        .withColumn("monetary_percentile", percent_rank().over(Window.orderBy(desc("monetary_total_value"))))
                 .withColumn("recency_score", 
                   when(col("recency_percentile") <= 0.2, 5)
                   .when(col("recency_percentile") <= 0.4, 4)
                   .when(col("recency_percentile") <= 0.6, 3)
                   .when(col("recency_percentile") <= 0.8, 2)
                   .otherwise(1))
        .withColumn("frequency_score",
                   when(col("frequency_percentile") >= 0.8, 5)
                   .when(col("frequency_percentile") >= 0.6, 4)
                   .when(col("frequency_percentile") >= 0.4, 3)
                   .when(col("frequency_percentile") >= 0.2, 2)
                   .otherwise(1))
        .withColumn("monetary_score",
                   when(col("monetary_percentile") >= 0.8, 5)
                   .when(col("monetary_percentile") >= 0.6, 4)
                   .when(col("monetary_percentile") >= 0.4, 3)
                   .when(col("monetary_percentile") >= 0.2, 2)
                   .otherwise(1))
    )
    
    return (
        customers.alias("c")
        .filter(col("c.__END_AT").isNull())
        .join(rfm_with_scores.alias("rfm"), col("c.customer_id") == col("rfm.customer_id"), "left")
        .join(balance_behavior.alias("bb"), col("c.customer_id") == col("bb.customer_id"), "left")
        .join(risk_metrics.alias("rm"), col("c.customer_id") == col("rm.customer_id"), "left")
        .select(
            col("c.customer_id"),
            col("c.jmeno"),
            col("c.prijmeni"),
            col("c.client_age"),
            col("c.prijem").alias("annual_income"),
            col("c.days_from_open_first_acc").alias("customer_tenure_days"),
            
            # RFM Scores
            coalesce(col("rfm.recency_score"), lit(1)).alias("recency_score"),
            coalesce(col("rfm.frequency_score"), lit(1)).alias("frequency_score"),
            coalesce(col("rfm.monetary_score"), lit(1)).alias("monetary_score"),
            
            # RFM Raw Values
            coalesce(col("rfm.recency_days"), lit(999)).alias("recency_days"),
            coalesce(col("rfm.frequency_transactions"), lit(0)).alias("frequency_transactions"),
            coalesce(col("rfm.monetary_total_value"), lit(0)).alias("monetary_total_value"),
            coalesce(col("rfm.monetary_avg_transaction"), lit(0)).alias("monetary_avg_transaction"),
            
            # Behavioral Metrics
            coalesce(col("rfm.category_diversity"), lit(0)).alias("spending_category_diversity"),
            coalesce(col("rfm.merchant_diversity"), lit(0)).alias("merchant_diversity"),
            
            # Seasonal Spending Patterns
            coalesce(col("rfm.winter_spending"), lit(0)).alias("winter_spending"),
            coalesce(col("rfm.spring_spending"), lit(0)).alias("spring_spending"),
            coalesce(col("rfm.summer_spending"), lit(0)).alias("summer_spending"),
            coalesce(col("rfm.autumn_spending"), lit(0)).alias("autumn_spending"),
            
            # Time Behavior
            coalesce(col("rfm.business_hours_txns"), lit(0)).alias("business_hours_transactions"),
            coalesce(col("rfm.evening_txns"), lit(0)).alias("evening_transactions"),
            coalesce(col("rfm.night_txns"), lit(0)).alias("night_transactions"),
            
            # Channel Preference
            coalesce(col("rfm.card_transactions"), lit(0)).alias("card_transactions"),
            coalesce(col("rfm.financial_transactions"), lit(0)).alias("financial_transactions"),
            
            # Balance Behavior
            coalesce(col("bb.avg_balance_365d"), lit(0)).alias("avg_balance_365d"),
            coalesce(col("bb.balance_volatility"), lit(0)).alias("balance_volatility"),
            coalesce(col("bb.negative_balance_days"), lit(0)).alias("negative_balance_days"),
            
            # Risk Metrics
            coalesce(col("rm.composite_risk_score"), lit(5)).alias("risk_score"),
            coalesce(col("rm.risk_category"), lit("Unknown")).alias("risk_category"),
            
            CURRENT_TIMESTAMP.alias("calculated_at")
        )
        .withColumn("rfm_composite_score", 
                   col("recency_score") + col("frequency_score") + col("monetary_score"))
        .withColumn("rfm_segment",
                   when(col("rfm_composite_score") >= 13, "Champions")
                   .when(col("rfm_composite_score") >= 11, "Loyal Customers")
                   .when(col("rfm_composite_score") >= 9, "Potential Loyalists")
                   .when(col("rfm_composite_score") >= 7, "New Customers")
                   .when(col("rfm_composite_score") >= 5, "Promising")
                   .when(col("rfm_composite_score") >= 3, "Customers Needing Attention")
                   .otherwise("Cannot Lose Them"))
        .withColumn("seasonal_preference",
                   when((col("winter_spending") >= col("spring_spending")) & 
                         (col("winter_spending") >= col("summer_spending")) & 
                         (col("winter_spending") >= col("autumn_spending")), "Winter Spender")
                   .when((col("spring_spending") >= col("winter_spending")) & 
                         (col("spring_spending") >= col("summer_spending")) & 
                         (col("spring_spending") >= col("autumn_spending")), "Spring Spender")
                   .when((col("summer_spending") >= col("winter_spending")) & 
                         (col("summer_spending") >= col("spring_spending")) & 
                         (col("summer_spending") >= col("autumn_spending")), "Summer Spender")
                   .when((col("autumn_spending") >= col("winter_spending")) & 
                         (col("autumn_spending") >= col("spring_spending")) & 
                         (col("autumn_spending") >= col("summer_spending")), "Autumn Spender")
                   .otherwise("Balanced"))
        .withColumn("time_behavior_segment",
                   when(col("business_hours_transactions") > (col("evening_transactions") + col("night_transactions")), "Business Hours User")
                   .when(col("evening_transactions") > (col("business_hours_transactions") + col("night_transactions")), "Evening User")
                   .when(col("night_transactions") > (col("business_hours_transactions") + col("evening_transactions")), "Night User")
                   .otherwise("Balanced User"))
        .withColumn("channel_preference",
                   when(col("card_transactions") > col("financial_transactions") * 2, "Card Dominant")
                   .when(col("financial_transactions") > col("card_transactions") * 2, "Financial Dominant")
                   .otherwise("Balanced Channel"))
        .withColumn("balance_stability_score",
                   when(col("balance_volatility") < 1000, 5)
                   .when(col("balance_volatility") < 5000, 4)
                   .when(col("balance_volatility") < 10000, 3)
                   .when(col("balance_volatility") < 20000, 2)
                   .otherwise(1))
        .withColumn("customer_value_tier",
                   when((col("monetary_score") >= 4) & (col("frequency_score") >= 4) & (col("avg_balance_365d") > 50000), "Premium")
                   .when((col("monetary_score") >= 3) & (col("frequency_score") >= 3) & (col("avg_balance_365d") > 20000), "Gold")
                   .when((col("monetary_score") >= 2) & (col("frequency_score") >= 2) & (col("avg_balance_365d") > 5000), "Silver")
                   .otherwise("Standard"))
        .withColumn("churn_risk_score",
                   when((col("recency_days") > 90) & (col("frequency_transactions") < 5), 5)  # High risk
                   .when((col("recency_days") > 60) & (col("frequency_transactions") < 10), 4)
                   .when((col("recency_days") > 30) & (col("frequency_transactions") < 20), 3)
                   .when(col("recency_days") > 14, 2)
                   .otherwise(1))  # Low risk
    )


In [ ]:
# ===============================================================================
#                           EXECUTIVE ANALYTICS VIEWS
# ===============================================================================

#------------------------------- EXECUTIVE DASHBOARD KPIs -------------------------
@dlt.table(
    name="view_executive_dashboard_kpis",
    comment="Real-time executive KPIs and performance indicators for C-level decision making",
    table_properties={
        "quality": "gold",
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true"
    }
)
def view_executive_dashboard_kpis():
    daily_bi = dlt.readStream(f"{catalog}.{gold_schema}.agg_daily_business_intelligence")
    clv_facts = dlt.readStream(f"{catalog}.{gold_schema}.fact_customer_lifetime_value")
    customer_seg = dlt.readStream(f"{catalog}.{gold_schema}.agg_customer_segmentation_advanced")
    
    # Get latest business date metrics
    latest_daily_metrics = (
        daily_bi
        .withColumn("row_num", row_number().over(Window.orderBy(desc("business_date"))))
        .filter(col("row_num") == 1)
        .drop("row_num")
    )
    
    # Customer portfolio summary
    customer_portfolio = (
        customer_seg
        .agg(
            count("*").alias("total_customers"),
            sum(when(col("customer_value_tier") == "Premium", 1).otherwise(0)).alias("premium_customers"),
            sum(when(col("customer_value_tier") == "Gold", 1).otherwise(0)).alias("gold_customers"),
            sum(when(col("customer_value_tier") == "Silver", 1).otherwise(0)).alias("silver_customers"),
            sum(when(col("customer_value_tier") == "Standard", 1).otherwise(0)).alias("standard_customers"),
            
            sum(when(col("churn_risk_score") >= 4, 1).otherwise(0)).alias("high_churn_risk_customers"),
            sum(when(col("rfm_segment") == "Champions", 1).otherwise(0)).alias("champion_customers"),
            sum(when(col("rfm_segment") == "Cannot Lose Them", 1).otherwise(0)).alias("at_risk_customers"),
            
            avg("monetary_total_value").alias("avg_customer_annual_value"),
            sum("monetary_total_value").alias("total_portfolio_value")
        )
    )
    
    # Revenue and profitability metrics
    revenue_metrics = (
        clv_facts
        .agg(
            sum("estimated_annual_revenue").alias("total_estimated_annual_revenue"),
            avg("estimated_annual_revenue").alias("avg_customer_annual_revenue"),
            sum("lifetime_value_score").alias("total_portfolio_clv"),
            avg("lifetime_value_score").alias("avg_customer_clv"),
            
            sum(when(col("customer_tier") == "Premium", col("estimated_annual_revenue")).otherwise(0)).alias("premium_tier_revenue"),
            sum(when(col("customer_tier") == "Gold", col("estimated_annual_revenue")).otherwise(0)).alias("gold_tier_revenue"),
            sum(when(col("customer_tier") == "Silver", col("estimated_annual_revenue")).otherwise(0)).alias("silver_tier_revenue"),
            sum(when(col("customer_tier") == "Standard", col("estimated_annual_revenue")).otherwise(0)).alias("standard_tier_revenue"),
            
            count(when(col("churn_risk_level") == "High Churn Risk", 1)).alias("high_churn_risk_count"),
            sum(when(col("churn_risk_level") == "High Churn Risk", col("estimated_annual_revenue")).otherwise(0)).alias("at_risk_revenue")
        )
    )
    
    return (
        latest_daily_metrics.alias("dm")
        .crossJoin(customer_portfolio.alias("cp"))
        .crossJoin(revenue_metrics.alias("rm"))
        .select(
            col("dm.business_date").alias("report_date"),
            
            # Operational KPIs
            col("dm.total_transactions").alias("daily_transaction_volume"),
            col("dm.total_transaction_value").alias("daily_transaction_value"),
            col("dm.active_customers").alias("daily_active_customers"),
            col("dm.revenue_per_active_customer").alias("daily_revenue_per_customer"),
            
            # Portfolio Composition
            col("cp.total_customers"),
            col("cp.premium_customers"),
            col("cp.gold_customers"),
            col("cp.silver_customers"),
            col("cp.standard_customers"),
            
            # Customer Health Metrics
            col("cp.champion_customers"),
            col("cp.high_churn_risk_customers"),
            col("cp.at_risk_customers"),
            round((col("cp.high_churn_risk_customers") / col("cp.total_customers")) * 100, 2).alias("churn_risk_percentage"),
            
            # Financial Performance
            round(col("rm.total_estimated_annual_revenue"), 2).alias("total_estimated_annual_revenue"),
            round(col("rm.avg_customer_annual_revenue"), 2).alias("avg_customer_annual_revenue"),
            round(col("rm.total_portfolio_clv"), 2).alias("total_portfolio_clv"),
            round(col("rm.avg_customer_clv"), 2).alias("avg_customer_clv"),
            
            # Revenue Distribution
            round(col("rm.premium_tier_revenue"), 2).alias("premium_tier_revenue"),
            round(col("rm.gold_tier_revenue"), 2).alias("gold_tier_revenue"),
            round(col("rm.silver_tier_revenue"), 2).alias("silver_tier_revenue"),
            round(col("rm.standard_tier_revenue"), 2).alias("standard_tier_revenue"),
            
            # Risk Metrics
            col("rm.high_churn_risk_count"),
            round(col("rm.at_risk_revenue"), 2).alias("revenue_at_risk"),
            round((col("rm.at_risk_revenue") / col("rm.total_estimated_annual_revenue")) * 100, 2).alias("revenue_at_risk_percentage"),
            
            # Balance Sheet Indicators
            col("dm.total_customer_balances"),
            col("dm.avg_customer_balance"),
            col("dm.negative_balance_accounts"),
            col("dm.balance_concentration_ratio"),
            
            # Operational Efficiency
            col("dm.transaction_efficiency_score"),
            col("dm.risk_indicator_score"),
            round(col("dm.total_transaction_value") / col("dm.total_transactions"), 2).alias("avg_transaction_size"),
            
            # Growth Indicators
            round((col("cp.premium_customers") + col("cp.gold_customers")) / col("cp.total_customers") * 100, 2).alias("high_value_customer_percentage"),
            round(col("cp.avg_customer_annual_value"), 2).alias("avg_customer_portfolio_value"),
            
            CURRENT_TIMESTAMP.alias("calculated_at")
        )
        .withColumn("portfolio_health_score",
                   when((col("churn_risk_percentage") < 10) & (col("high_value_customer_percentage") > 30), "Excellent")
                   .when((col("churn_risk_percentage") < 20) & (col("high_value_customer_percentage") > 20), "Good")
                   .when((col("churn_risk_percentage") < 30) & (col("high_value_customer_percentage") > 10), "Fair")
                   .otherwise("Poor"))
        .withColumn("revenue_concentration_risk",
                   when(col("premium_tier_revenue") / col("total_estimated_annual_revenue") > 0.5, "High")
                   .when(col("premium_tier_revenue") / col("total_estimated_annual_revenue") > 0.3, "Medium")
                   .otherwise("Low"))
    )


In [ ]:
#------------------------------- PREDICTIVE ANALYTICS VIEW -------------------------
@dlt.table(
    name="view_predictive_analytics_insights",
    comment="Advanced predictive analytics for strategic decision making, including trend analysis and forecasting",
    table_properties={
        "quality": "gold",
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true"
    }
)
def view_predictive_analytics_insights():
    transaction_analytics = dlt.readStream(f"{catalog}.{gold_schema}.fact_transaction_analytics_advanced")
    daily_bi = dlt.readStream(f"{catalog}.{gold_schema}.agg_daily_business_intelligence")
    customer_seg = dlt.readStream(f"{catalog}.{gold_schema}.agg_customer_segmentation_advanced")
    
    # Transaction trend analysis (last 90 days)
    transaction_trends = (
        transaction_analytics
        .filter(col("transaction_date") >= date_sub(CURRENT_DATE, 90))
        .groupBy(
            weekofyear("transaction_date").alias("week_of_year"),
            year("transaction_date").alias("year")
        )
        .agg(
            count("*").alias("weekly_transaction_count"),
            sum("amount").alias("weekly_transaction_value"),
            avg("amount").alias("weekly_avg_transaction"),
            countDistinct("customer_id").alias("weekly_active_customers"),
            
            # Fraud indicators
            sum(when(col("fraud_risk_level") == "High Risk", 1).otherwise(0)).alias("weekly_high_risk_transactions"),
            sum(when(col("anomaly_score") >= 3, 1).otherwise(0)).alias("weekly_anomalous_transactions"),
            
            # Behavioral patterns
            sum(when(col("customer_behavior_segment") == "High Value High Frequency", 1).otherwise(0)).alias("weekly_hvhf_customers"),
            avg("transaction_velocity_score").alias("weekly_avg_velocity_score")
        )
        .withColumn("week_over_week_growth",
                   (col("weekly_transaction_value") - 
                    lag("weekly_transaction_value", 1).over(Window.orderBy("year", "week_of_year"))) /
                   lag("weekly_transaction_value", 1).over(Window.orderBy("year", "week_of_year")) * 100)
        .withColumn("customer_growth_rate",
                   (col("weekly_active_customers") - 
                    lag("weekly_active_customers", 1).over(Window.orderBy("year", "week_of_year"))) /
                   lag("weekly_active_customers", 1).over(Window.orderBy("year", "week_of_year")) * 100)
    )
    
    # Customer lifecycle predictions
    customer_lifecycle_predictions = (
        customer_seg
        .select(
            "customer_id",
            "customer_value_tier",
            "rfm_segment", 
            "churn_risk_score",
            "recency_days",
            "frequency_transactions",
            "monetary_total_value",
            "balance_stability_score"
        )
                 .withColumn("predicted_next_action",
                   when((col("churn_risk_score") >= 4) & (col("customer_value_tier").isin(["Premium", "Gold"])), "Immediate Retention Campaign")
                   .when((col("churn_risk_score") >= 3) & (col("recency_days") > 30), "Engagement Campaign")
                   .when((col("rfm_segment") == "Potential Loyalists") & (col("monetary_total_value") > 10000), "Upsell Campaign")
                   .when((col("rfm_segment") == "New Customers") & (col("frequency_transactions") > 10), "Loyalty Program Enrollment")
                   .when(col("balance_stability_score") <= 2, "Financial Health Check")
                   .otherwise("Standard Monitoring"))
        .withColumn("predicted_ltv_trend",
                   when((col("frequency_transactions") > 50) & (col("monetary_total_value") > 20000) & (col("recency_days") < 14), "Increasing")
                   .when((col("churn_risk_score") >= 4) | (col("recency_days") > 60), "Decreasing")
                   .when((col("frequency_transactions") > 20) & (col("recency_days") < 30), "Stable Growth")
                   .otherwise("Stable"))
        .withColumn("engagement_propensity",
                   when((col("frequency_transactions") > 30) & (col("recency_days") < 7), "High")
                   .when((col("frequency_transactions") > 10) & (col("recency_days") < 30), "Medium")
                   .otherwise("Low"))
    )
    
    # Market opportunity analysis
    market_opportunities = (
        customer_seg
        .groupBy("customer_value_tier", "seasonal_preference", "channel_preference")
        .agg(
            count("*").alias("segment_size"),
            avg("monetary_total_value").alias("avg_segment_value"),
            sum("monetary_total_value").alias("total_segment_value"),
            avg("churn_risk_score").alias("avg_churn_risk"),
            avg("balance_stability_score").alias("avg_stability")
        )
        .withColumn("segment_attractiveness_score",
                   round((col("avg_segment_value") * col("segment_size") * col("avg_stability")) / 
                         (col("avg_churn_risk") + 1), 2))
                 .withColumn("growth_potential",
                   when(col("segment_attractiveness_score") > 100000, "High Potential")
                   .when(col("segment_attractiveness_score") > 50000, "Medium Potential")
                   .otherwise("Low Potential"))
    )
    
    # Risk and compliance predictions
    risk_predictions = (
        transaction_analytics
        .filter(col("transaction_date") >= date_sub(CURRENT_DATE, 30))
        .groupBy("customer_id")
        .agg(
            sum(when(col("fraud_risk_level") == "High Risk", 1).otherwise(0)).alias("recent_high_risk_transactions"),
            avg("anomaly_score").alias("avg_anomaly_score"),
            max("anomaly_score").alias("max_anomaly_score"),
            count(when(col("is_night_transaction") == 1, 1)).alias("night_transactions_30d"),
            countDistinct("transaction_region").alias("geographic_diversity_30d")
        )
                 .withColumn("fraud_risk_prediction",
                   when((col("recent_high_risk_transactions") >= 3) | (col("max_anomaly_score") >= 5), "High Risk")
                   .when((col("recent_high_risk_transactions") >= 1) | (col("avg_anomaly_score") >= 3), "Medium Risk")
                   .otherwise("Low Risk"))
        .withColumn("compliance_alert_level",
                   when((col("night_transactions_30d") > 10) & (col("geographic_diversity_30d") > 5), "Enhanced Monitoring")
                   .when(col("recent_high_risk_transactions") > 0, "Standard Monitoring")
                   .otherwise("Normal"))
    )
    
    # Aggregate all insights
    current_trends = (
        transaction_trends
        .withColumn("row_num", row_number().over(Window.orderBy(desc("year"), desc("week_of_year"))))
        .filter(col("row_num") <= 4)  # Last 4 weeks
        .agg(
            avg("weekly_transaction_count").alias("avg_weekly_transactions"),
            avg("weekly_transaction_value").alias("avg_weekly_value"),
            avg("week_over_week_growth").alias("avg_growth_rate"),
            avg("weekly_high_risk_transactions").alias("avg_weekly_risk_transactions"),
            max("weekly_hvhf_customers").alias("peak_hvhf_customers")
        )
    )
    
    lifecycle_summary = (
        customer_lifecycle_predictions
        .agg(
            count(when(col("predicted_next_action") == "Immediate Retention Campaign", 1)).alias("immediate_retention_needed"),
            count(when(col("predicted_next_action") == "Upsell Campaign", 1)).alias("upsell_opportunities"),
            count(when(col("predicted_ltv_trend") == "Increasing", 1)).alias("growing_value_customers"),
            count(when(col("predicted_ltv_trend") == "Decreasing", 1)).alias("declining_value_customers"),
            count(when(col("engagement_propensity") == "High", 1)).alias("high_engagement_customers")
        )
    )
    
    opportunity_summary = (
        market_opportunities
        .filter(col("growth_potential") == "High Potential")
        .agg(
            count("*").alias("high_potential_segments"),
            sum("total_segment_value").alias("high_potential_value"),
            max("segment_attractiveness_score").alias("max_attractiveness_score")
        )
    )
    
    risk_summary = (
        risk_predictions
        .agg(
            count(when(col("fraud_risk_prediction") == "High Risk", 1)).alias("high_fraud_risk_customers"),
            count(when(col("compliance_alert_level") == "Enhanced Monitoring", 1)).alias("enhanced_monitoring_customers"),
            avg("avg_anomaly_score").alias("overall_avg_anomaly_score")
        )
    )
    
    return (
        current_trends.alias("ct")
        .crossJoin(lifecycle_summary.alias("ls"))
        .crossJoin(opportunity_summary.alias("os"))
        .crossJoin(risk_summary.alias("rs"))
        .select(
            # Trend Analysis
            round(col("ct.avg_weekly_transactions"), 0).alias("avg_weekly_transaction_volume"),
            round(col("ct.avg_weekly_value"), 2).alias("avg_weekly_transaction_value"),
            round(col("ct.avg_growth_rate"), 2).alias("average_growth_rate_percent"),
            
            # Customer Lifecycle Predictions
            col("ls.immediate_retention_needed"),
            col("ls.upsell_opportunities"),
            col("ls.growing_value_customers"),
            col("ls.declining_value_customers"),
            col("ls.high_engagement_customers"),
            
            # Market Opportunities
            coalesce(col("os.high_potential_segments"), lit(0)).alias("high_potential_market_segments"),
            round(coalesce(col("os.high_potential_value"), lit(0)), 2).alias("high_potential_segment_value"),
            round(coalesce(col("os.max_attractiveness_score"), lit(0)), 2).alias("max_segment_attractiveness"),
            
            # Risk Intelligence
            col("rs.high_fraud_risk_customers"),
            col("rs.enhanced_monitoring_customers"),
            round(col("rs.overall_avg_anomaly_score"), 2).alias("portfolio_anomaly_score"),
            round(col("ct.avg_weekly_risk_transactions"), 0).alias("avg_weekly_risk_transactions"),
            
                         # Strategic Recommendations
             when(col("ct.avg_growth_rate") > 5, "Aggressive Growth Strategy")
             .when(col("ct.avg_growth_rate") > 0, "Steady Growth Strategy")
             .when(col("ct.avg_growth_rate") > -5, "Stabilization Strategy")
             .otherwise("Recovery Strategy").alias("recommended_business_strategy"),
             
             when(col("ls.immediate_retention_needed") > 100, "Critical")
             .when(col("ls.immediate_retention_needed") > 50, "High")
             .when(col("ls.immediate_retention_needed") > 20, "Medium")
             .otherwise("Low").alias("retention_priority_level"),
             
             when(col("rs.high_fraud_risk_customers") > 50, "Enhanced Security Protocols")
             .when(col("rs.high_fraud_risk_customers") > 20, "Increased Monitoring")
             .otherwise("Standard Security").alias("security_recommendation"),
            
            # Performance Indicators
            round((col("ls.growing_value_customers") / 
                  (col("ls.growing_value_customers") + col("ls.declining_value_customers") + 1)) * 100, 2).alias("customer_value_growth_ratio"),
            
            round((col("ls.upsell_opportunities") / 
                  (col("ls.immediate_retention_needed") + 1)) * 100, 2).alias("opportunity_to_risk_ratio"),
            
            CURRENT_TIMESTAMP.alias("analysis_timestamp"),
            CURRENT_DATE.alias("analysis_date")
        )
    )
